# QTran：Carinthia 五投影正式实验

本 Notebook 使用第二轮冻结配置，对 Carinthia 的四个参数匹配模型进行正式评估。由于最少类别只有4个样本，普通 70/15/15 分层切分无法保证每个集合都有全部类别，因此这里固定使用4折外层测试，并在每个外层训练分区内按类别划分验证集。

## 固定协议

- 每个外层测试折都包含全部6类，最少类别每折恰好1个测试样本；
- 外层测试样本不参与训练、早停或模型选择；
- 每个模型使用5个训练种子，每个种子训练4折，共 `4模型 × 5种子 × 4折 = 80` 个任务；
- 每个模型和种子的四折测试预测按原始样本编号拼接，得到覆盖全部4591张图一次的 OOF 结果；
- 论文主表使用5个种子级 OOF 指标，不把四折误当成独立重复实验；
- 每个任务独立保存并支持断点续跑。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from qcs_core import plot_result_bars
from qcs_frozen_benchmark import (
    FORMAL_SEEDS,
    audit_carinthia_formal,
    formal_progress,
    load_frozen_selection,
    require_complete,
    run_carinthia_formal,
)
from qcs_multidataset import paired_model_comparisons

pd.set_option('display.max_columns', 100)
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_frozen_benchmark.py').exists():
    raise RuntimeError('请从 /root/xxx/autodl 目录打开并运行本 Notebook')
PROJECT_DIR

## 1. 路径与冻结文件检查

In [ ]:
CACHE_PATH = PROJECT_DIR / 'data_cache' / 'carinthia_32.npz'
FROZEN_SELECTION = (
    PROJECT_DIR
    / 'artifacts'
    / 'stability_search_stage2'
    / 'mixedwm38'
    / 'frozen_stage2_selection.json'
)
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'three_datasets_five_projection'

for path in (CACHE_PATH, FROZEN_SELECTION):
    if not path.exists():
        raise FileNotFoundError(path)

frozen = load_frozen_selection(FROZEN_SELECTION)
display(pd.DataFrame([{
    'cache': str(CACHE_PATH),
    'frozen_selection': str(FROZEN_SELECTION),
    'artifact_root': str(ARTIFACT_ROOT),
    'selected_candidate_id': frozen.selected_candidate_id,
    'frozen_sha256': frozen.sha256,
}]))

## 2. 四折协议审计

重点检查最少类别是否在每折的训练、验证和测试部分都存在，以及四模型参数量是否仍在公平范围内。

In [ ]:
audit = audit_carinthia_formal(
    CACHE_PATH,
    FROZEN_SELECTION,
    split_seed=2026,
)
display(audit['frozen'])
display(pd.DataFrame([audit['dataset']]))
display(audit['classes'])
display(audit['folds'].pivot_table(
    index=['outer_fold', 'label'],
    columns='split',
    values='count',
))
display(audit['parameters'])

## 3. 查看断点续跑状态

In [ ]:
progress_before = formal_progress(ARTIFACT_ROOT, 'carinthia', FORMAL_SEEDS)
display(progress_before.groupby('model')['complete'].agg(['sum', 'count']))
print(f"已完成 {int(progress_before['complete'].sum())}/{len(progress_before)} 个任务")

## 4. 运行或续跑80个正式任务

`MAX_JOBS=None` 表示运行所有剩余任务。建议首次可设为 `1` 检查一个任务；确认正常后改回 `None`。程序每完成一折就保存一次，服务器中断后重新执行本单元即可继续。

In [ ]:
MAX_JOBS = None

carinthia_oof, carinthia_folds = run_carinthia_formal(
    cache_path=CACHE_PATH,
    frozen_selection_path=FROZEN_SELECTION,
    artifact_dir=ARTIFACT_ROOT,
    seeds=FORMAL_SEEDS,
    split_seed=2026,
    resume=True,
    max_jobs=MAX_JOBS,
)
print(f'当前已收集 {len(carinthia_folds)}/80 个折任务')
print(f'当前已形成 {len(carinthia_oof)}/20 个完整种子级 OOF 结果')
display(carinthia_oof)

## 5. 完整性检查与 OOF 主结果

只有80/80个折任务完成后，才把 OOF 汇总用于论文模型比较。

In [ ]:
progress_after = formal_progress(ARTIFACT_ROOT, 'carinthia', FORMAL_SEEDS)
display(progress_after.groupby('model')['complete'].agg(['sum', 'count']))
require_complete(progress_after, 'carinthia')

OOF_PATH = ARTIFACT_ROOT / 'carinthia' / 'carinthia_oof_results.csv'
carinthia_oof = pd.read_csv(OOF_PATH)
carinthia_summary = pd.read_csv(ARTIFACT_ROOT / 'carinthia' / 'summary.csv')
display(carinthia_summary)

carinthia_paired = paired_model_comparisons(
    carinthia_oof,
    metric='macro_f1',
)
carinthia_paired.to_csv(
    ARTIFACT_ROOT / 'carinthia' / 'paired_macro_f1.csv',
    index=False,
)
display(carinthia_paired)

In [ ]:
figure = plot_result_bars(carinthia_oof, metric='macro_f1')
figure.savefig(
    ARTIFACT_ROOT / 'carinthia' / 'formal_oof_macro_f1.png',
    dpi=300,
    bbox_inches='tight',
)
plt.show()

## 解释限制

最少类别总共只有4张图，因此该类别的召回率高度离散，单次误分类就会显著改变结果。论文应同时报告每类样本数、每类召回率、混淆矩阵和这一限制，不应把 Carinthia 单独作为“量子优势”的证据。完成后保留整个 `artifacts/three_datasets_five_projection/carinthia/` 文件夹。